In [9]:
# Install dependencies (uv-managed venv, no pip by default)
!uv add anthropic python-dotenv


Resolved 51 packages in 10ms
Audited 47 packages in 9ms


In [10]:
# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()


True

In [11]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [14]:

# add a message to the conversation
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=0.5, stop_sequences=None):
    params = {"model": model, "messages": messages, "max_tokens": 1000, "temperature": temperature, "stop_sequences": stop_sequences}

    if system:
        params["system"] = system
    message = client.messages.create(
        **params
    )
    text_blocks = [block.text for block in message.content if block.type == "text"]
    if not text_blocks:
        return ""  # or raise a meaningful error
    return text_blocks[0]


In [15]:
messages = []

add_user_message(messages, "Count from 1 to 10")

chat(messages, stop_sequences=["5","3,4"])



'1\n2\n3\n4\n'

In [18]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])
text

'\n{\n  "Name": "MyEventRule",\n  "EventBusName": "default",\n  "EventPattern": {\n    "source": ["aws.ec2"],\n    "detail-type": ["EC2 Instance State-change Notification"],\n    "detail": {\n      "state": ["running"]\n    }\n  },\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",\n      "Id": "1"\n    }\n  ]\n}\n'

In [22]:
import json

# Clean up and parse the JSON
clean_json = json.loads(text.strip())
clean_json

{'Name': 'MyEventRule',
 'EventBusName': 'default',
 'EventPattern': {'source': ['aws.ec2'],
  'detail-type': ['EC2 Instance State-change Notification'],
  'detail': {'state': ['running']}},
 'State': 'ENABLED',
 'Targets': [{'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:MyFunction',
   'Id': '1'}]}